# Task 9 -- The "Date Drift" Amortization Audit

## Overview

LLMs are notoriously poor at **calendar-aware financial arithmetic**.
When asked to build an amortization table they tend to:
- Assume **30 days per month** (30/360 convention) regardless of actual calendar,
- **Miss leap-day (Feb 29)**, collapsing February to 28 days, and
- Produce a "textbook-smooth" table that does **not** match actual daily-accrual balances.

This notebook:
1. Computes the **ground-truth** amortization using Python `datetime` (exact calendar days).
2. Sends a **naive prompt** to the LLM and captures its answer.
3. Sends a **guided chain-of-thought prompt** that forces the LLM to reason about actual dates.
4. Compares all three and visualises the **divergence**.

---

### Loan Parameters
| Parameter | Value |
|---|---|
| Principal | $100,000 |
| Annual Rate | 12% |
| Day-count | Actual / 365 |
| Start Date | 1 Jan 2024 (leap year) |

### Payment Log
| Date | Payment |
|---|---|
| 31 Jan 2024 | $5,000 |
| 29 Feb 2024 | $5,000 (Leap Day!) |
| 31 Mar 2024 | $5,000 |

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'matplotlib', 'seaborn']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("Dependencies ready")

In [ ]:
import os, json, requests
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
print("Imports OK")

---
## Part 1 -- Ground-Truth Calculation (Python / `datetime`)

Using `datetime.date` we compute the **exact number of calendar days** between each payment,
including the leap day on 29 Feb 2024.

Interest formula:
> **Interest = Balance x (0.12 / 365) x Days_Elapsed**

In [ ]:
# Loan parameters
PRINCIPAL   = 100_000.00
ANNUAL_RATE = 0.12
DAILY_RATE  = ANNUAL_RATE / 365

start_date     = date(2024, 1, 1)
payment_dates  = [date(2024, 1, 31),
                  date(2024, 2, 29),   # Leap Day
                  date(2024, 3, 31)]
payment_amount = 5_000.00

# Amortization engine
rows    = []
balance = PRINCIPAL
prev_dt = start_date

for pmt_date in payment_dates:
    days_elapsed  = (pmt_date - prev_dt).days
    interest      = balance * DAILY_RATE * days_elapsed
    balance_due   = balance + interest
    balance_after = balance_due - payment_amount

    rows.append({
        'Payment Date'        : pmt_date.strftime('%d %b %Y'),
        'Days Elapsed'        : days_elapsed,
        'Opening Balance ($)' : round(balance, 4),
        'Interest Accrued ($)': round(interest, 4),
        'Balance Due ($)'     : round(balance_due, 4),
        'Payment ($)'         : payment_amount,
        'Closing Balance ($)' : round(balance_after, 4),
    })

    prev_dt = pmt_date
    balance = balance_after

df_gt = pd.DataFrame(rows)

print("=" * 70)
print(" GROUND-TRUTH AMORTIZATION TABLE  (Actual/365 day count)")
print("=" * 70)
print(df_gt.to_string(index=False))
print(f"\n  Final closing balance: ${df_gt['Closing Balance ($)'].iloc[-1]:,.4f}")

### Key observations
- **Jan 1 -> Jan 31 = 30 days** (straightforward)
- **Jan 31 -> Feb 29 = 29 days** -- 2024 is a leap year; an LLM may assume 28 (non-leap) or 30 (30/360)
- **Feb 29 -> Mar 31 = 31 days** -- an LLM may round to 30

---
## Part 2 -- LLM Client Setup

We use **Groq** (free tier) with LLaMA 3.3 70B.
Get a free key at **console.groq.com -> API Keys**.

In [ ]:
os.environ['GROQ_API_KEY'] = 'paste-your-groq-key-here'   # replace this

MODEL = 'llama-3.3-70b-versatile'

def ask_llm(system: str, user: str, max_tokens: int = 1200) -> str:
    """Call Groq API using plain requests -- no openai package needed."""
    headers = {
        'Authorization': f'Bearer {os.environ["GROQ_API_KEY"]}',
        'Content-Type': 'application/json',
    }
    payload = {
        'model': MODEL,
        'max_tokens': max_tokens,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
    }
    resp = requests.post(
        'https://api.groq.com/openai/v1/chat/completions',
        headers=headers,
        data=json.dumps(payload),
        timeout=60
    )
    resp.raise_for_status()
    return resp.json()['choices'][0]['message']['content']

print(f"LLM client ready  |  model = {MODEL}")

---
## Part 3 -- Naive LLM Prompt (The Trap)

This prompt gives the LLM only the raw loan parameters -- no hints about exact day counts.
We expect it to assume 30-day months or ignore the leap day.

In [ ]:
SYSTEM_NAIVE = """You are a financial analyst. Produce a precise amortisation table.
Use Actual/365 daily interest: Interest = Balance x (0.12 / 365) x Days_Elapsed.
Show each row: Payment Date | Days Elapsed | Opening Balance | Interest | Balance Due | Payment | Closing Balance.
Round all dollar amounts to 4 decimal places."""

USER_NAIVE = """Loan details:
  Principal:   $100,000
  Annual rate: 12%  (daily: balance x 0.12/365 x days_elapsed)
  Start date:  January 1, 2024

Payments of $5,000 made on: January 31, February 29, March 31, 2024.

Build the amortisation table and state the final closing balance."""

print("Sending naive prompt to LLM...")
naive_response = ask_llm(SYSTEM_NAIVE, USER_NAIVE)
print("\n" + "=" * 70)
print(" LLM RESPONSE -- NAIVE PROMPT")
print("=" * 70)
print(naive_response)

---
## Part 4 -- Guided Chain-of-Thought Prompt

This prompt explicitly supplies the **exact day counts** and instructs step-by-step reasoning.
This is the correct prompting strategy to avoid date drift.

In [ ]:
SYSTEM_GUIDED = """You are a meticulous loan auditor. Follow the user's instructions exactly.
Use the exact day counts provided -- do NOT substitute 30-day months or ignore the leap day.
Interest formula: Interest = Balance x (0.12 / 365) x Exact_Days.
Compute step-by-step, rounding dollar amounts to 4 decimal places at each step."""

USER_GUIDED = """Audit this loan. 2024 is a LEAP YEAR.

Loan: $100,000 at 12% p.a., daily: Balance x (0.12 / 365) x Exact_Days.
Start: 1 Jan 2024.

Payment schedule with EXACT calendar days:
  1. 31 Jan 2024  -- exactly 30 days since 1 Jan     -- pay $5,000
  2. 29 Feb 2024  -- exactly 29 days since 31 Jan (Feb has 29 days in 2024!) -- pay $5,000
  3. 31 Mar 2024  -- exactly 31 days since 29 Feb    -- pay $5,000

For each payment:
  Step A: State exact days elapsed.
  Step B: Compute interest = opening_balance x (0.12 / 365) x days.
  Step C: Compute balance_due = opening_balance + interest.
  Step D: Compute closing_balance = balance_due - 5000.

Finish with the final closing balance after all three payments."""

print("Sending guided prompt to LLM...")
guided_response = ask_llm(SYSTEM_GUIDED, USER_GUIDED, max_tokens=1400)
print("\n" + "=" * 70)
print(" LLM RESPONSE -- GUIDED CHAIN-OF-THOUGHT PROMPT")
print("=" * 70)
print(guided_response)

---
## Part 5 -- Parse LLM Numbers & Compare

We extract the closing balances reported by each LLM response and compare with ground truth.

In [ ]:
import re

def extract_balances(text: str) -> list:
    candidates = re.findall(r'\$?([0-9]{2,3},[0-9]{3}(?:\.[0-9]+)?)', text)
    vals = []
    for c in candidates:
        v = float(c.replace(',', ''))
        if 80_000 <= v <= 100_000:
            vals.append(round(v, 4))
    seen, unique = set(), []
    for v in vals:
        key = round(v, 0)
        if key not in seen:
            seen.add(key)
            unique.append(v)
    return unique[:3]

gt_balances     = df_gt['Closing Balance ($)'].tolist()
naive_balances  = extract_balances(naive_response)
guided_balances = extract_balances(guided_response)

labels = ['After Jan 31', 'After Feb 29', 'After Mar 31']

def pad(lst, n=3):
    return lst + [None] * (n - len(lst))

df_compare = pd.DataFrame({
    'Period'           : labels,
    'Ground Truth ($)' : gt_balances,
    'LLM Naive ($)'    : pad(naive_balances),
    'LLM Guided ($)'   : pad(guided_balances),
})
df_compare['Naive Error ($)']  = (df_compare['LLM Naive ($)']  - df_compare['Ground Truth ($)']).round(4)
df_compare['Guided Error ($)'] = (df_compare['LLM Guided ($)'] - df_compare['Ground Truth ($)']).round(4)

print("=" * 80)
print(" COMPARISON: Ground Truth vs LLM Naive vs LLM Guided")
print("=" * 80)
print(df_compare.to_string(index=False))

---
## Part 6 -- Visualisations

In [ ]:
# Fig 1: Closing balance comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Task 9 -- Date-Drift Amortisation Audit', fontsize=14, fontweight='bold')

x, width = range(len(labels)), 0.28
ax = axes[0]
ax.bar([i - width for i in x], df_compare['Ground Truth ($)'], width, label='Ground Truth', color='steelblue')
ax.bar([i         for i in x], df_compare['LLM Naive ($)'],    width, label='LLM Naive',    color='tomato',   alpha=0.85)
ax.bar([i + width for i in x], df_compare['LLM Guided ($)'],   width, label='LLM Guided',   color='seagreen', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(labels, fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.set_title('Closing Balances After Each Payment')
ax.set_ylabel('Balance ($)'); ax.legend(fontsize=9); ax.set_ylim(85_000, 100_000)

ax2 = axes[1]
naive_errs  = df_compare['Naive Error ($)'].fillna(0).abs().tolist()
guided_errs = df_compare['Guided Error ($)'].fillna(0).abs().tolist()
ax2.bar([i - width/2 for i in x], naive_errs,  width, label='Naive Error',  color='tomato',   alpha=0.85)
ax2.bar([i + width/2 for i in x], guided_errs, width, label='Guided Error', color='seagreen', alpha=0.85)
ax2.set_xticks(list(x)); ax2.set_xticklabels(labels, fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.2f}'))
ax2.set_title('Absolute Error vs Ground Truth')
ax2.set_ylabel('|Error| ($)'); ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('amortization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Fig 2: Balance trajectory
fig2, ax3 = plt.subplots(figsize=(10, 5))

all_dates    = [start_date.strftime('%d %b %Y')] + df_gt['Payment Date'].tolist()
all_balances = [PRINCIPAL] + df_gt['Closing Balance ($)'].tolist()

ax3.plot(all_dates, all_balances, marker='o', linewidth=2.2, color='steelblue', zorder=3)
for d, b in zip(all_dates, all_balances):
    ax3.annotate(f'${b:,.2f}', xy=(d, b), xytext=(0, 10),
                 textcoords='offset points', ha='center', fontsize=8.5, color='navy')

day_counts = [30, 29, 31]
for i, dc in enumerate(day_counts):
    mid_y = (all_balances[i] + all_balances[i+1]) / 2
    ax3.annotate(f'{dc} days', xy=(i + 0.5, mid_y), xytext=(20, 0),
                 textcoords='offset points', fontsize=9, color='firebrick')

ax3.set_title('Ground-Truth Balance Trajectory (Actual/365)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Outstanding Balance ($)')
ax3.set_xlabel('Date')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax3.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('balance_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 7 -- Day-Count Sensitivity Analysis

How much does the final balance change depending on which Feb day-count the LLM assumes?

In [ ]:
def compute_final_balance(feb_days: int) -> float:
    bal = PRINCIPAL
    for dc in [30, feb_days, 31]:
        bal = bal + bal * DAILY_RATE * dc - 5_000
    return round(bal, 4)

sensitivity = pd.DataFrame({
    'Feb Days Assumed' : [28, 29, 30],
    'Assumption'       : ['LLM non-leap (wrong)', 'Actual / Correct', 'LLM 30-day month (wrong)'],
    'Final Balance ($)': [compute_final_balance(d) for d in [28, 29, 30]],
})
gt_final = compute_final_balance(29)
sensitivity['Error vs Correct ($)'] = (sensitivity['Final Balance ($)'] - gt_final).round(4)

print("=" * 65)
print(" SENSITIVITY: Impact of Different Feb Day-Count Assumptions")
print("=" * 65)
print(sensitivity.to_string(index=False))

fig3, ax4 = plt.subplots(figsize=(8, 4))
colors = ['tomato', 'seagreen', 'tomato']
bars = ax4.bar(sensitivity['Assumption'], sensitivity['Final Balance ($)'],
               color=colors, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, sensitivity['Final Balance ($)']):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'${val:,.2f}', ha='center', va='bottom', fontsize=9)
ax4.set_title('Final Balance vs Feb Day-Count Assumption', fontweight='bold')
ax4.set_ylabel('Final Closing Balance ($)')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax4.set_ylim(sensitivity['Final Balance ($)'].min() - 100,
             sensitivity['Final Balance ($)'].max() + 300)
plt.xticks(fontsize=8)
plt.tight_layout()
plt.savefig('sensitivity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 8 -- Conclusions

### What the Analysis Shows

| Issue | Impact |
|---|---|
| Treating Feb 2024 as 28 days (non-leap assumption) | Understates interest |
| Treating Feb as 30 days (30/360 assumption) | Overstates interest |
| Using correct 29 days (Actual/365) | Correct result |

### LLM Prompting Lessons

1. **Naive prompts fail** -- even when the formula is stated, LLMs default to 30/360 or ignore leap years.
2. **Supply explicit day counts** -- providing exact calendar days eliminates date-drift errors.
3. **Chain-of-thought guidance** -- step-by-step reasoning per payment period improves numerical accuracy.
4. **Never trust LLM calendar math unchecked** -- always validate with Python `datetime` as the authoritative source.

### Verdict on the Borrower's Claim
If the bank used correct Actual/365 with proper leap-day recognition, any deviation (e.g. 30-day months)
constitutes a **pricing error** and should be flagged in the audit.
> *Ground-truth Python calculation is the authoritative reference; LLM outputs must be treated as advisory only.*

In [ ]:
final_gt = df_gt['Closing Balance ($)'].iloc[-1]
print("=" * 50)
print(f"  AUTHORITATIVE FINAL BALANCE: ${final_gt:,.4f}")
print("=" * 50)
print(f"  Total interest paid : ${sum(df_gt['Interest Accrued ($)']):,.4f}")
print(f"  Total payments made : ${3 * 5000:,.2f}")
print(f"  Principal repaid    : ${PRINCIPAL - final_gt:,.4f}")